# dataloader-batching composite — cx23: DataLoader iterates batches, scalar-reduced loss backward per batch

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `dataloader-batching`, `backward-on-scalar-loss`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, TensorDataset

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "dataloader-batching"
DD_ATOM_IDS = ["dataloader-batching", "backward-on-scalar-loss"]
DD_SUBTOPICS = ["PyTorch: DataLoader batching", "PyTorch: backward()"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The minimal real-data training step has two pieces:

1. **DataLoader batching.** Wrap a `TensorDataset` in a `DataLoader(batch_size=B)` and iterate. Each iteration yields a tuple `(x_batch, y_batch)` of shape `(B, ...)`. The loader knows to slice the dataset into chunks of size `B` and shuffle if asked.
2. **Scalar-reduced backward.** Compute a per-sample loss vector of shape `(B,)`, reduce to a scalar via `.mean()`, then `loss.backward()` to populate `param.grad`. Per-batch grads are *accumulated* if you don't zero between batches.

**Anatomy.**
```python
loader = DataLoader(TensorDataset(x, y), batch_size=B)   # dataloader-batching.
for xb, yb in loader:
    per_sample = ((param * xb) - yb) ** 2                # shape (B,) ish.
    loss = per_sample.mean()                             # backward-on-scalar-loss.
    loss.backward()                                       # accumulates into param.grad.
```

### Composite Exercise — DataLoader iterates batches, scalar-reduced loss backward per batch

**Atoms exercised together**: `dataloader-batching`, `backward-on-scalar-loss`

Implement `cx23_dataloader_train(x_data, y_data, param, batch_size)`.

Inputs:
- `x_data` — `t.Tensor` of shape `(N,)`, the inputs.
- `y_data` — `t.Tensor` of shape `(N,)`, the targets.
- `param` — a scalar `t.Tensor` (shape `()`) with `requires_grad=True`. The 'model' is `y_hat = param * x`.
- `batch_size` — int.

Required behaviour:
1. Build a `TensorDataset(x_data, y_data)`.
2. Wrap in `DataLoader(ds, batch_size=batch_size, shuffle=False)` (atom: dataloader-batching).
3. Initialise `losses = []`.
4. For each `(xb, yb)` in the loader:
   - Compute `per_sample = (param * xb - yb) ** 2` (shape `(B,)`).
   - Reduce to a scalar via `.mean()` (atom: backward-on-scalar-loss).
   - Call `.backward()`.
   - Append the scalar loss as a Python float to `losses`.
5. Return `losses`.

Do **NOT** zero grads between batches — the test relies on grad accumulating across all batches (so `param.grad` at the end equals the SUM of per-batch grads, demonstrating accumulation semantics).

Test checks:
- `losses` has the right length (`ceil(N / batch_size)`).
- Each loss is a Python float (proves you reduced to scalar before `.item()`).
- `param.grad` accumulates across batches.
- The total accumulated grad equals what you'd get from one big batch — `.mean()` per batch then summed across batches DOES NOT equal one big `.mean()` unless batches are equal-size, but the SUM of `.sum()` would. The test uses an N divisible by batch_size to keep the math clean.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx23_dataloader_train(x_data, y_data, param, batch_size):
    """Iterate a DataLoader, scalar-reduce per-batch loss, backward. Returns list of batch losses."""
    raise NotImplementedError

def _test_cx23():
    # Case A: simple — N=8, batch_size=4 → 2 batches, 2 losses.
    t.manual_seed(0)
    x = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
    y = 2.0 * x   # true param = 2.
    param = t.tensor(0.5, requires_grad=True)
    losses = cx23_dataloader_train(x, y, param, batch_size=4)
    assert isinstance(losses, list)
    assert len(losses) == 2, f'N=8, B=4 → 2 batches; got {len(losses)} losses'
    for L in losses:
        assert isinstance(L, float), f'each loss should be a Python float (proves scalar reduction); got {type(L).__name__}'
    assert param.grad is not None, 'param.grad must be populated after backward'

    # Case B: grad accumulates — final grad equals sum of per-batch grads.
    # We can recompute the expected grad: each batch contributes mean of d/dparam (param*x - y)^2.
    # Total grad = sum_{batches} mean_b ( 2*(param*x - y)*x ).
    param2 = t.tensor(0.5, requires_grad=True)
    expected_grad = 0.0
    x_batches = [x[0:4], x[4:8]]
    y_batches = [y[0:4], y[4:8]]
    for xb, yb in zip(x_batches, y_batches):
        g_batch = (2 * (param2.item() * xb - yb) * xb).mean().item()
        expected_grad += g_batch
    losses2 = cx23_dataloader_train(x, y, param2, batch_size=4)
    assert abs(param2.grad.item() - expected_grad) < 1e-5, (
        f'expected accumulated grad {expected_grad:.5f}; got {param2.grad.item():.5f}'
    )

    # Case C: batch_size=1 → N batches of size 1 each.
    x3 = t.tensor([1.0, 2.0, 3.0])
    y3 = t.tensor([2.0, 4.0, 6.0])
    param3 = t.tensor(0.0, requires_grad=True)
    losses3 = cx23_dataloader_train(x3, y3, param3, batch_size=1)
    assert len(losses3) == 3, f'B=1 should give N=3 batches; got {len(losses3)}'
    assert param3.grad is not None

    # Case D: batch_size > N → 1 batch with all data (DataLoader's last-batch behaviour).
    x4 = t.tensor([1.0, 2.0, 3.0])
    y4 = t.tensor([2.0, 4.0, 6.0])
    param4 = t.tensor(0.0, requires_grad=True)
    losses4 = cx23_dataloader_train(x4, y4, param4, batch_size=10)
    assert len(losses4) == 1, f'B=10 > N=3 should give exactly 1 batch; got {len(losses4)}'

    # Case E: loss values are roughly right magnitude.
    # For param=0, y=2x → per-sample loss = (0 - 2x)^2 = 4x^2.
    # Batch 1 (x=1,2,3): mean(4 + 16 + 36) = 56/3 ≈ 18.67.
    assert abs(losses4[0] - 56.0 / 3.0) < 1e-4, f'expected ~{56/3:.4f}; got {losses4[0]:.4f}'

    # Case F: backward populates grad even when batches have unequal sizes (last partial batch).
    x5 = t.arange(7.0)   # N=7.
    y5 = 3.0 * x5
    param5 = t.tensor(0.0, requires_grad=True)
    losses5 = cx23_dataloader_train(x5, y5, param5, batch_size=3)
    # N=7, B=3 → batches of sizes 3, 3, 1.
    assert len(losses5) == 3, f'N=7 B=3 → 3 batches (3+3+1); got {len(losses5)}'
    assert param5.grad is not None
    assert not t.isnan(param5.grad), 'grad must not be NaN for unequal batch sizes'
    _dd_passed.add('cx23')

_test_cx23()

<details><summary>Show solution — cx23</summary>

```python
def cx23_dataloader_train(x_data, y_data, param, batch_size):
    # Atom A (dataloader-batching): wrap tensors in TensorDataset → DataLoader.
    ds = TensorDataset(x_data, y_data)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    losses = []
    for xb, yb in loader:
        # Per-sample squared error, shape (B,).
        per_sample = (param * xb - yb) ** 2
        # Atom B (backward-on-scalar-loss): reduce to scalar, then backward.
        loss = per_sample.mean()
        loss.backward()
        losses.append(loss.item())
    return losses
```

`shuffle=False` is required for the test's grad-accumulation check to compare against a deterministic per-batch slicing of the data. In a real trainer you'd `shuffle=True` and call `optimizer.zero_grad()` BEFORE each batch's backward — that's the topic of cx22 / cx24. The composition isolated here is just 'data → batches → scalar backward'.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx23'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx23',
        'subtopics': ["PyTorch: DataLoader batching", "PyTorch: backward()"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()